# EinsteinEngine: Cosmological Perfect Fluid & Benchmark

In this test, we model a flat, expanding universe using the FLRW metric.
We will:
1. Benchmark pure SymPy vs EinsteinEngine performance.
2. Compute the Einstein Tensor ($G_{\mu\nu}$) representing the spacetime geometry.
3. Compute the Energy-Momentum Tensor ($T_{\mu\nu}$) representing the cosmic fluid.
4. Extract the Friedmann Equations by equating $G_{\mu\nu} = \kappa T_{\mu\nu}$.

In [15]:
import time
import sympy as sp
from einsteinengine.symbolic.metric import MetricTensor
from einsteinengine.symbolic.einstein_tensor import EinsteinTensor
from einsteinengine.symbolic.energy_momentum import EnergyMomentumTensor

sp.init_printing(use_latex='mathjax') #for better display of equations in Jupyter Notebook

# 1. Define spacetime coordinates and physical functions
t, x, y, z = sp.symbols('t x y z')
syms = [t, x, y, z]

# a(t) is the scale factor of the universe (how it expands over time)
a = sp.Function('a')(t)

# 2. Define the flat FLRW metric (ds^2 = -dt^2 + a(t)^2 (dx^2 + dy^2 + dz^2))
g_flrw = [
    [-1, 0, 0, 0],
    [0, a**2, 0, 0],
    [0, 0, a**2, 0],
    [0, 0, 0, a**2]
]


In [22]:
print("\n--- BENCHMARK: EINSTEIN ENGINE (C++ BACKEND) ---")

start_time_engine = time.time()

# Instantiate the metric
metric_flrw = MetricTensor(g_flrw, syms, name="FLRW", verbose=False)

# Compute the entire geometric pipeline up to the Einstein Tensor
# This calculates Christoffel, Riemann, Ricci Tensor, and Ricci Scalar internally
G_tensor = EinsteinTensor.from_metric(metric_flrw, verbose=False)

# Deep simplify the result to collapse the equations
G_tensor.simplify(in_place=True)

engine_duration = time.time() - start_time_engine
print(f"EinsteinEngine completed FULL Einstein Tensor pipeline in: {engine_duration:.5f} seconds")


--- BENCHMARK: EINSTEIN ENGINE (C++ BACKEND) ---
EinsteinEngine completed FULL Einstein Tensor pipeline in: 0.19898 seconds


In [23]:
import time
import sympy as sp

# Assuming g_flrw and syms (t, x, y, z) are already defined in previous cells

print("\n--- BENCHMARK: PURE SYMPY FULL PIPELINE (BRUTE FORCE) ---")
print("Warning: This may take several seconds. SymPy is calculating 337 tensor components sequentially...\n")

start_time_pure = time.time()

# 1. Inverse Metric
g_matrix = sp.Matrix(g_flrw)
g_inv = g_matrix.inv().tolist()

# 2. Christoffel Symbols (Gamma^lam_{mu nu}) - 64 components
Gamma = [[[sp.sympify(0) for _ in range(4)] for _ in range(4)] for _ in range(4)]
for lam in range(4):
    for mu in range(4):
        for nu in range(4):
            tmp = 0
            for sig in range(4):
                term1 = sp.diff(g_flrw[nu][sig], syms[mu])
                term2 = sp.diff(g_flrw[mu][sig], syms[nu])
                term3 = sp.diff(g_flrw[mu][nu], syms[sig])
                tmp += g_inv[lam][sig] * (term1 + term2 - term3)
            # Simplification is forced at each step to prevent memory explosion in later tensors
            Gamma[lam][mu][nu] = sp.simplify(sp.Rational(1, 2) * tmp)

# 3. Riemann Tensor (R^rho_{sig mu nu}) - 256 components
Riemann = [[[[sp.sympify(0) for _ in range(4)] for _ in range(4)] for _ in range(4)] for _ in range(4)]
for rho in range(4):
    for sig in range(4):
        for mu in range(4):
            for nu in range(4):
                term1 = sp.diff(Gamma[rho][nu][sig], syms[mu])
                term2 = sp.diff(Gamma[rho][mu][sig], syms[nu])
                term3 = sum(Gamma[rho][mu][lam] * Gamma[lam][nu][sig] for lam in range(4))
                term4 = sum(Gamma[rho][nu][lam] * Gamma[lam][mu][sig] for lam in range(4))
                Riemann[rho][sig][mu][nu] = sp.simplify(term1 - term2 + term3 - term4)

# 4. Ricci Tensor (R_{mu nu} = R^lam_{mu lam nu}) - 16 components
Ricci = [[sp.sympify(0) for _ in range(4)] for _ in range(4)]
for mu in range(4):
    for nu in range(4):
        Ricci[mu][nu] = sp.simplify(sum(Riemann[lam][mu][lam][nu] for lam in range(4)))

# 5. Ricci Scalar (R = g^{mu nu} R_{mu nu}) - 1 component
Ricci_scalar = sp.simplify(sum(g_inv[mu][nu] * Ricci[mu][nu] for mu in range(4) for nu in range(4)))

# 6. Einstein Tensor (G_{mu nu} = R_{mu nu} - 1/2 g_{mu nu} R) - 16 components
Einstein_pure = [[sp.sympify(0) for _ in range(4)] for _ in range(4)]
for mu in range(4):
    for nu in range(4):
        Einstein_pure[mu][nu] = sp.simplify(Ricci[mu][nu] - sp.Rational(1, 2) * g_flrw[mu][nu] * Ricci_scalar)

pure_duration = time.time() - start_time_pure
print(f"Pure SymPy completed FULL Einstein Tensor pipeline in: {pure_duration:.5f} seconds")

# Verify that the results match
print("\n[VERIFICATION] Cross-checking pure SymPy G_00 with EinsteinEngine G_00...")
engine_G00 = G_tensor.get_raw_data()[0][0]
pure_G00 = Einstein_pure[0][0]

if sp.simplify(engine_G00 - pure_G00) == 0:
    print("Match confirmed! Both engines produced the exact same physical result.")
else:
    print("Warning: Mismatch detected.")


--- BENCHMARK: PURE SYMPY FULL PIPELINE (BRUTE FORCE) ---

Pure SymPy completed FULL Einstein Tensor pipeline in: 0.55785 seconds

[VERIFICATION] Cross-checking pure SymPy G_00 with EinsteinEngine G_00...
Match confirmed! Both engines produced the exact same physical result.


In [18]:
from IPython.display import display
print("\n--- PHYSICS TEST: MATTER & FIELD EQUATIONS ---")

# 1. Define physical parameters for the Perfect Fluid
rho = sp.Function('rho')(t) # Cosmic energy density
p = sp.Function('p')(t)     # Cosmic pressure
u_mu = [-1, 0, 0, 0]        # Comoving observer (at rest with the fluid)

# 2. Build the Energy-Momentum Tensor using the factory method
T_tensor = EnergyMomentumTensor.from_perfect_fluid(
    metric=metric_flrw, 
    density=rho, 
    pressure=p, 
    four_velocity_cov=u_mu, 
    verbose=True
)

# 3. Display the resulting T_mu_nu matrix
print("\n[RESULT] Energy-Momentum Matrix (T_mu_nu):")
for row in T_tensor.get_raw_data():
    print(row)

# 4. Extract the First Friedmann Equation (Time-Time component 0,0)
# G_{00} = 8 * pi * G * T_{00} (Using kappa = 8 * pi)
kappa = sp.symbols('kappa')
G_00 = G_tensor.get_raw_data()[0][0]
T_00 = T_tensor.get_raw_data()[0][0]

friedmann_eq = sp.Eq(G_00, kappa * T_00)

print("\n[RESULT] First Friedmann Equation (Derived automatically):")
display(friedmann_eq)


--- PHYSICS TEST: MATTER & FIELD EQUATIONS ---
Building Perfect Fluid Energy-Momentum Tensor for 'FLRW'...
[Fluid_FLRW] Instantiated in 4D spacetime.
[Fluid_FLRW] Tensor initialized with index configuration: 'll'.

[RESULT] Energy-Momentum Matrix (T_mu_nu):
[rho(t), 0, 0, 0]
[0, a(t)**2*p(t), 0, 0]
[0, 0, a(t)**2*p(t), 0]
[0, 0, 0, a(t)**2*p(t)]

[RESULT] First Friedmann Equation (Derived automatically):


            2         
  ⎛d       ⎞          
3⋅⎜──(a(t))⎟          
  ⎝dt      ⎠          
───────────── = κ⋅ρ(t)
     2                
    a (t)             

In [19]:
# 1. Define the concrete formula for a(t) (e.g., Matter-dominated universe)
a_concrete = t**(sp.Rational(2, 3))

# 2. Substitute the concrete a(t) into our Friedmann Equation
# We replace the abstract Function 'a(t)' with our concrete expression
friedmann_with_a = friedmann_eq.subs(a, a_concrete).doit()

# 3. Solve algebraically for rho(t)
rho_solved = sp.solve(friedmann_with_a, rho)[0]

print("If the universe expands as a(t) = t^(2/3), the density must clear as:")
display(sp.Eq(rho, rho_solved))

If the universe expands as a(t) = t^(2/3), the density must clear as:


         4   
ρ(t) = ──────
            2
       3⋅κ⋅t 

In [20]:
# 1. Define a positive density constant for Dark Energy
rho_0 = sp.symbols('rho_0', positive=True)

# 2. Substitute rho(t) with our constant in the Friedmann Equation
friedmann_ode = friedmann_eq.subs(rho, rho_0)

print("Differential Equation to solve:")
display(friedmann_ode)

print("\nSolving the differential equation for a(t)... (SymPy is working)")
# 3. sp.dsolve is SymPy's engine for Ordinary Differential Equations (ODEs)
# We ask it to solve for the scale factor function a(t)
solutions = sp.dsolve(friedmann_ode, a)

print("\n[RESULT] Solutions for the expansion of the Universe:")
for sol in solutions:
    display(sol)

Differential Equation to solve:


            2       
  ⎛d       ⎞        
3⋅⎜──(a(t))⎟        
  ⎝dt      ⎠        
───────────── = κ⋅ρ₀
     2              
    a (t)           


Solving the differential equation for a(t)... (SymPy is working)

[RESULT] Solutions for the expansion of the Universe:


                    ____   
           -√3⋅√κ⋅╲╱ ρ₀ ⋅t 
           ────────────────
                  3        
a(t) = C₁⋅ℯ                

                   ____  
           √3⋅√κ⋅╲╱ ρ₀ ⋅t
           ──────────────
                 3       
a(t) = C₁⋅ℯ              

In [21]:
# 1. Define a constant for the initial radiation density (right after the Big Bang)
rho_0 = sp.symbols('rho_0', positive=True)

# 2. Define the dynamic density: radiation dilutes as the inverse fourth power of the scale factor
radiation_density = rho_0 / (a**4)

# 3. Substitute this dynamic density into our abstract Friedmann Equation
friedmann_rad = friedmann_eq.subs(rho, radiation_density)

print("Non-linear Differential Equation for a Radiation-Dominated Universe:")
display(friedmann_rad)

print("\nSolving the ODE for a(t)... (SymPy is doing the heavy lifting)")
# 4. Solve the coupled differential equation
solutions_rad = sp.dsolve(friedmann_rad, a)

print("\n[RESULT] Solutions for the scale factor a(t) in the early universe:")
for sol in solutions_rad:
    display(sol)

Non-linear Differential Equation for a Radiation-Dominated Universe:


            2        
  ⎛d       ⎞         
3⋅⎜──(a(t))⎟         
  ⎝dt      ⎠    κ⋅ρ₀ 
───────────── = ─────
     2           4   
    a (t)       a (t)


Solving the ODE for a(t)... (SymPy is doing the heavy lifting)

[RESULT] Solutions for the scale factor a(t) in the early universe:


           _______________________ 
          ╱                ____    
       -╲╱  C₁ - 6⋅√3⋅√κ⋅╲╱ ρ₀ ⋅t  
a(t) = ────────────────────────────
                    3              

          _______________________
         ╱                ____   
       ╲╱  C₁ - 6⋅√3⋅√κ⋅╲╱ ρ₀ ⋅t 
a(t) = ──────────────────────────
                   3             

           _______________________ 
          ╱                ____    
       -╲╱  C₁ + 6⋅√3⋅√κ⋅╲╱ ρ₀ ⋅t  
a(t) = ────────────────────────────
                    3              

          _______________________
         ╱                ____   
       ╲╱  C₁ + 6⋅√3⋅√κ⋅╲╱ ρ₀ ⋅t 
a(t) = ──────────────────────────
                   3             

Only the last solution is valid for an universe in expansion